# PHerc0211 Local Inference

Run the local `v8-in` checkpoint in this repo on the copied PHerc0211 TIFF stack at `data/training_surfaces/20260928000003/layers/`.

This notebook does not assemble zarrs. It reads the 24 local TIFF layers directly, uses the local mask for the crop, and writes a single fast figure to `output/test_visualizations/`.

In [ ]:
import os
from pathlib import Path

def _find_repo():
    env = os.getenv("VESUVIUS_REPO")
    candidates = []
    if env:
        candidates.append(Path(env))
    candidates.extend([Path.cwd(), Path.cwd().parent, Path('/vesuvius/nader-fork')])
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / 'model.safetensors').is_file() and (candidate / 'ink8um').is_dir():
            return candidate
    raise FileNotFoundError('Could not find nader-fork repo root')

REPO = _find_repo()
SEGMENT_ID = '20260928000003'
SEGMENT_NAME = 'PHerc0211_z7312_w080_abf'
SURFACE_DIR = REPO / 'data' / 'training_surfaces' / SEGMENT_ID
LAYERS_DIR = SURFACE_DIR / 'layers'
MASK_PATH = SURFACE_DIR / 'labels' / 'mask.png'
LABEL_PATH = SURFACE_DIR / 'labels' / 'inklabels.png'
MODEL_DIR = Path(os.getenv('VESUVIUS_V8_DIR', str(REPO)))
MODEL_PATH = MODEL_DIR / 'model.safetensors'
CONFIG_PATH = MODEL_DIR / 'config.json'
RUN_ID = 'v8in_base_local'
OUTPUT_DIR = REPO / 'output' / 'test_visualizations' / RUN_ID
DRY_RUN = False

print(REPO)
print(LAYERS_DIR)
print(MODEL_PATH)

In [ ]:
import json

if not MODEL_PATH.is_file() or not CONFIG_PATH.is_file():
    raise FileNotFoundError(f'missing local checkpoint under {MODEL_DIR}')
if not LAYERS_DIR.is_dir():
    raise FileNotFoundError(f'missing TIFF layers: {LAYERS_DIR}')
if not MASK_PATH.is_file():
    raise FileNotFoundError(f'missing mask: {MASK_PATH}')

with open(CONFIG_PATH, 'r', encoding='utf-8') as handle:
    V8_CONFIG = json.load(handle)

V8_DEPTH = int(V8_CONFIG['in_depth'])
V8_TILE = int(V8_CONFIG['tile_size'])
V8_CLIP = int(V8_CONFIG['clip_max'])
EVAL_STRIDE = int(V8_CONFIG['stride'])
REVERSE_LAYERS = False
BATCH_SIZE = 16
NUM_WORKERS = 4
MASK_CROP_MARGIN = 32
COMPOSITE_D0 = 10
COMPOSITE_D1 = 18
VOXEL_UM = 9.362

print(f'depth={V8_DEPTH} tile={V8_TILE} clip={V8_CLIP} stride={EVAL_STRIDE}')

In [ ]:
import sys
import gc
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
from safetensors.torch import load_file as load_safetensors

try:
    import huggingface_hub  # noqa: F401
    import safetensors  # noqa: F401
except ImportError:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'safetensors', 'huggingface_hub>=0.25'], check=True)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.chdir(REPO)
from ink8um import InkDetector
from ink8um.inference import coverage_mask, predict_stack, tile_positions

try:
    get_ipython().run_line_magic('matplotlib', 'agg')
except Exception:
    plt.switch_backend('agg')

Image.MAX_IMAGE_PIXELS = None
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PRED_CMAP = plt.get_cmap('gray_r').copy()
PRED_CMAP.set_bad((0.78, 0.86, 0.95))

def load_model():
    model = InkDetector(**V8_CONFIG)
    model.load_state_dict(load_safetensors(str(MODEL_PATH)), strict=True)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    return model.to(device).eval(), device

def _mask_bbox(mask_bool, margin):
    ys, xs = np.where(mask_bool)
    if ys.size == 0:
        return 0, mask_bool.shape[0], 0, mask_bool.shape[1]
    y0 = max(0, int(ys.min()) - margin)
    x0 = max(0, int(xs.min()) - margin)
    y1 = min(mask_bool.shape[0], int(ys.max()) + 1 + margin)
    x1 = min(mask_bool.shape[1], int(xs.max()) + 1 + margin)
    return y0, y1, x0, x1

def load_stack():
    layers = []
    for idx in range(V8_DEPTH):
        path = LAYERS_DIR / f'{idx:02d}.tif'
        if not path.is_file():
            raise FileNotFoundError(path)
        layer = np.array(Image.open(path).convert('L'))
        layers.append(np.clip(layer, 0, V8_CLIP).astype(np.uint8))
    return np.stack(layers, axis=-1)

def make_composite(stack, bounds):
    y0, y1, x0, x1 = bounds
    return np.max(stack[y0:y1, x0:x1, COMPOSITE_D0:COMPOSITE_D1], axis=-1).astype(np.uint8)

def predict_map(model, device, stack, bounds):
    prediction = predict_stack(
        model,
        stack,
        reverse=REVERSE_LAYERS,
        stride=EVAL_STRIDE,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        device=device,
    )
    covered = np.zeros(prediction.shape, bool)
    for x, y in tile_positions(coverage_mask(stack), V8_TILE, EVAL_STRIDE):
        covered[y:y + V8_TILE, x:x + V8_TILE] = True
    prediction[~covered] = np.nan
    y0, y1, x0, x1 = bounds
    return prediction[y0:y1, x0:x1]

def _draw_scale_bar(ax, panel_hw, px_scale):
    height, width = panel_hw
    length = 10000.0 / VOXEL_UM * px_scale
    thickness = max(2.0, min(height, width) / 250)
    pad = 0.05 * min(height, width) + thickness
    x1, y = width - pad, height - pad
    x0 = max(0.0, x1 - length)
    ax.add_patch(Rectangle((x0, y - thickness), x1 - x0, thickness, color='red'))
    ax.text(x0, y - 3 * thickness, '1 cm', color='red', fontsize=10, ha='left', va='bottom')

def render_segment():
    mask = np.array(Image.open(MASK_PATH).convert('L')) > 0
    bounds = _mask_bbox(mask, MASK_CROP_MARGIN)
    stack = load_stack()
    composite = make_composite(stack, bounds)
    if DRY_RUN:
        prediction = np.full(composite.shape, np.nan, np.float32)
    else:
        model, device = load_model()
        prediction = predict_map(model, device, stack, bounds)
    fig, axes = plt.subplots(2, 1, figsize=(12, 12), dpi=100, facecolor='white')
    axes[0].imshow(np.ma.masked_invalid(prediction), cmap=PRED_CMAP, vmin=0, vmax=1, interpolation='nearest')
    axes[0].set_title('v8-in prediction, centred 24 layers')
    axes[1].imshow(composite, cmap='gray', vmin=0, vmax=255, interpolation='nearest')
    axes[1].set_title('composite view')
    _draw_scale_bar(axes[1], composite.shape, 1.0)
    for ax in axes:
        ax.set_xticks([])
        ax.set_yticks([])
    out_path = OUTPUT_DIR / f'{SEGMENT_NAME}_fast.jpg'
    fig.savefig(out_path, dpi=100, facecolor='white', pil_kwargs={'quality': 92})
    plt.close(fig)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(out_path)

In [ ]:
render_segment()